# ISFL Play-by-Play Data Exploration

Explore the raw PBP and boxscore data structure from the ISFL index.

In [ ]:
from isfl_epa.scraper.pbp import fetch_pbp_file, fetch_game
from isfl_epa.scraper.boxscore import fetch_boxscore_file
from isfl_epa.config import League, get_engine
import json

## PBP Data Structure (2022 Engine - S27+)

In [ ]:
# Fetch a PBP file from Season 50
games = fetch_pbp_file(League.ISFL, 50, 1)
print(f'Games in file: {len(games)}')
game = games[0]
print(f'Game ID: {game["id"]}')
print(f'Top-level keys: {sorted(game.keys())}')

In [ ]:
# Play object fields
# Each play has: s (score), c (clock), t (down/distance), o (field position), m (play description), css (styling)
# The 'id' field on plays is the TEAM id (not game id)
play = game['Q1'][3]  # skip quarter header
print(json.dumps(play, indent=2))

In [ ]:
# Show all Q1 plays for a game
for i, play in enumerate(game['Q1']):
    print(f'{i:3d} | {play["c"]:>5} | {play["t"]:>15} | {play["o"]:>10} | {play["m"][:80]}')

## Field Meanings

| Field | Meaning | Example |
|-------|---------|--------|
| `id`  | Team ID (who has possession) | 6, 14 |
| `s`   | Current score | `NYS 0 - SJS 0` |
| `c`   | Game clock | `14:49` |
| `t`   | Down and distance | `1st and 10`, `3rd and 7` |
| `o`   | Field position | `SJS - 25`, `NYS - 48` |
| `m`   | Play description (the main data) | `Pass by Patterson, J., complete to...` |
| `css` | Display styling class | `c` (green/score), `d` (red/turnover), `e` (blue/special) |

In [ ]:
# Collect all unique play description patterns
import re
from collections import Counter

patterns = Counter()
for q in ['Q1', 'Q2', 'Q3', 'Q4', 'OT']:
    for play in game[q]:
        desc = play['m']
        # Extract the action verb/type
        if 'Pass by' in desc:
            patterns['Pass'] += 1
        elif 'Rush by' in desc:
            patterns['Rush'] += 1
        elif 'Kickoff by' in desc:
            patterns['Kickoff'] += 1
        elif 'Punt by' in desc:
            patterns['Punt'] += 1
        elif 'FG by' in desc:
            patterns['FG'] += 1
        elif 'Sacked' in desc or 'sacked' in desc:
            patterns['Sack'] += 1
        elif 'Quarter' in desc or 'Half' in desc:
            patterns['Quarter/Half marker'] += 1
        elif 'Penalty' in desc or 'penalty' in desc:
            patterns['Penalty'] += 1
        elif 'Timeout' in desc:
            patterns['Timeout'] += 1
        elif 'kneels' in desc:
            patterns['Kneel'] += 1
        elif 'kick' in desc.lower():
            patterns['Kick (other)'] += 1
        else:
            patterns[f'OTHER: {desc[:60]}'] += 1

for pat, count in patterns.most_common():
    print(f'{count:4d}  {pat}')

## Boxscore Data Structure

In [ ]:
boxscores = fetch_boxscore_file(League.ISFL, 50, 1)
bs = boxscores[0]
print(f'Boxscore keys: {sorted(bs.keys())}')

In [ ]:
# Team-level stats
print(f'{bs["aAbb"]} ({bs["aMascot"]}) vs {bs["hAbb"]} ({bs["hMascot"]})')
print(f'Final: {bs["aAbb"]} {bs["aF"]} - {bs["hAbb"]} {bs["hF"]}')
print(f'Weather: {bs["weather"]}')
print()
print(f'{"Stat":>20} | {bs["aAbb"]:>8} | {bs["hAbb"]:>8}')
print('-' * 45)
for stat, ak, hk in [
    ('First Downs', 'aFD', 'hFD'),
    ('Total Yards', 'aYds', 'hYds'),
    ('Passing Yards', 'aPassing', 'hPassing'),
    ('Rushing Yards', 'aRushing', 'hRushing'),
    ('3rd Down', 'a3rdM', 'h3rdM'),
    ('Penalties', 'aPen', 'hPen'),
    ('Penalty Yards', 'aPenYds', 'hPenYds'),
    ('TOP', 'aTOP', 'hTOP'),
]:
    print(f'{stat:>20} | {str(bs.get(ak, "")):>8} | {str(bs.get(hk, "")):>8}')

In [ ]:
# Player stats from boxscore
for stat_key in ['aStatsPassing', 'aStatsRushing', 'aStatsReceiving', 'aStatsDef', 'aStatsKicking', 'aStatsPunting', 'aStatsST', 'aStatsOther']:
    val = bs.get(stat_key, [])
    if val:
        print(f'\n=== {stat_key} ({len(val)} players) ===')
        print(json.dumps(val[0], indent=2))

## Compare Earliest Season (S27) with Latest (S59)

In [ ]:
# Compare S27 vs S59 PBP format
for season in [27, 59]:
    games = fetch_pbp_file(League.ISFL, season, 1)
    game = games[0]
    print(f'\n=== Season {season} (engine: {get_engine(season).value}) ===')
    print(f'Game keys: {sorted(game.keys())}')
    # Show a few plays
    for play in game['Q1'][1:4]:
        print(f'  Play keys: {sorted(play.keys())}')
        print(f'  {play["m"][:100]}')
        break